<h1>Briggs-Haldane Model</h1>
<h3>Derivation of hyperbolic dependence of rate on substrate concentration</h3>
<p>G. Woehlke 2015 ff (converted from Wolfram Mathematica, aided by A.I.)</p>

<h3>Kinetic scheme: </h3> 
<img src="2-08_BriggsHaldane.png" width="466"> 
<p>The enzyme E binds substrate S reversibly with rate $k_{12}$ and releases it with $k_{21}$. Product formation is irreversible, its rate is $k_{23}$. It is usually assumed that $k_{23}$ is slow and rate-limiting.</p>
<p>We have the follwoing symbolic representation using <i>SymPy</i>, which states that the total (inital) concentration of enzyme $E_{total}$ is split into fractions of ligand-free enzyme $E(t)$ and ligand-bound $ES(t)$. The flow from E+S to ES is
    $\mathrm{d}c_{ES}(t)/\mathrm{dt} = +k_{12} c_{E}(t) c_{S}(t) -k_{21} c_{ES}(t) -k_{23} c_{ES}(t) $. The solution is simplified in the next step: </p>

In [1]:
%pip install -q -r /Users/guenther_work/Publication/MolecularMachines_Springer_2025/PythonCode/requirements.txt

Note: you may need to restart the kernel to use updated packages.


In [2]:
import sympy as sp

# symbols
k12, k21, k23 = sp.symbols('k12 k21 k23', positive=True)
cE, cES, cS = sp.symbols('cE cES cS', positive=True)
E_total = sp.symbols('E_total', positive=True)

# equations (steady state + conservation)
eqs = [
    sp.Eq(k12*cE*cS - k21*cES - k23*cES, 0),
    sp.Eq(E_total, cE + cES)
]

# solve
sol = sp.solve(eqs, (cES, cE))

sol

{cE: (E_total*k21 + E_total*k23)/(cS*k12 + k21 + k23),
 cES: E_total*cS*k12/(cS*k12 + k21 + k23)}

In [3]:
cES_expr = sp.simplify(sol[cES])
cE_expr  = sp.simplify(sol[cE])

cES_expr

E_total*cS*k12/(cS*k12 + k21 + k23)

<h3>Result: </h3>
<p>This means: <span style="color:blue;font-weight: bold">The observed reaction velocity</span> per enzyme is thus $v / c_{E,total} =  k_{23} \cdot c_S / \left( {\frac{k_{21}+k_{23}}{k_{12}} \cdot c_S} \right)$. It is proportional to the rate of product formation $k_{23}$, and depends hyperbolically on the substrate concentration $c_S$. The 'new' half-maximal velocity concentration is now $K_M = \frac{k_{21}+k_{23}}{k_{12}}$.</p>
<p>Note that we have to use $K_M$ "Michaelis-Menten constant" <span style="color:blue;font-weight: bold"> carefully</span>, in particular we always keep in mind that it differs from substrate affinity $K_S = \frac{k_{21}}{k_{12}}$! This is shown below. </p>
<p>To clearly distinguish substrate affinity and the kinetic parameter $K_M$ it has been suggested to speak of <span style="color:blue;font-weight: bold">"half-maximal activation concentration" $K_{1/2}$</span>, i.e. the concentration that produces 1/2 of the maximal turnover.</p>

In [4]:
v = sp.simplify(k23 * cES_expr)
v

E_total*cS*k12*k23/(cS*k12 + k21 + k23)

In [5]:
KM = sp.symbols('KM', positive=True)

v_MM = E_total * k23 * cS / (KM + cS)

# solve for KM
KM_expr = sp.solve(sp.Eq(v, v_MM), KM)[0]

sp.simplify(KM_expr)

(k21 + k23)/k12

<h3>Derivative of the inital slope</h3>
<p>To calculate the slope of the Briggs-Haldane function at substrate concentration zero ($c_{S}=0$), we form the derivative of this veocity function against $c$, $\mathrm{d}v/\mathrm{dc}$, and set $c_{S}=0$:</p>

In [6]:
dv_dcS = sp.diff(v_MM, cS)

sp.simplify(dv_dcS)

E_total*KM*k23/(KM + cS)**2

In [7]:
initial_slope = sp.simplify(dv_dcS.subs(cS, 0))
initial_slope

E_total*k23/KM

<h3>Result: </h3>
<p>The slope per enzyme (divide by $E_{total}$) is $k_{23}/K_M$. If we asssume that $k_{23}$ is rate-limiting and much slower than the other rate constants, this is approximately $k_{bi} = k_{cat}/K_{1/2}$, the <span style="color:blue;font-weight: bold">apparent bimolecular binding rate constant</span>.</p>

<h3>Calculation and plotting routine</h3>
<p>The following code plots y1, y2 as Briggs-Haldane curve (observed velocity = f(concentration), and the first derivative at point $c = 0$. A crosshair is placed at half-maximal turnover $\frac{1}{2}v_{max}$ and $K_{1/2} = \frac{k_{21}+k_{23}}{k_{12}}$.  </p>

In [8]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider


def plot_bh(linewidth, k12_val, k21_val, k23_val, rangeX=5, rangeY=1.2):
    
    x = np.linspace(0.01, rangeX, 300)
    
    # curves
    y1 = k12_val * x / (k21_val + k23_val)
    y2 = k12_val * x / (k21_val + k23_val + k12_val * x)
    y3 = np.ones_like(x)
    
    KM_val = (k21_val + k23_val) / k12_val
    
    plt.figure(figsize=(8,5))
    
    plt.plot(x, y1, label='initial slope', color='darkblue',linewidth=linewidth)
    plt.plot(x, y2, label='observed rate', color='blue', linewidth=linewidth)
    plt.plot(x, y3, '--', color='gray')
    
    # vertical KM line
    plt.axvline(KM_val, color='gray', linestyle='--')
    
    # key point
    plt.scatter([KM_val], [0.5], color='black')
    
    plt.xlim(0, rangeX)
    plt.ylim(0, rangeY)
    
    plt.xlabel('$cS /M$')
    plt.ylabel('$k_{obs} /s^{-1}$')
    plt.title('Briggs–Haldane scheme')
    
    plt.legend()
    plt.grid(True)
    
    plt.text(KM_val + 0.2, 0.25,
             f"K(1/2) = (k21 + k23)/k12 =  {(k21_val + k23_val)/k12_val:.2f} $M$\n"
             f"$k21/k12 ≈$ {k21_val/k12_val:.2f} $M$",
             fontsize=10)
    
    plt.text(KM_val + 0.2, 0.9,
             f"initial slope ≈ {k12_val/(k21_val + k23_val):.2f} $M^{-1}\cdot s^{-1}$",
             fontsize=10)
    
    plt.show()


from ipywidgets import interact, FloatSlider

interact(
    plot_bh,
    linewidth=(1,5),
    k12_val=FloatSlider(value=5.0, min=0.1, max=25.0, step=0.1, description='k12'),
    k21_val=FloatSlider(value=5.0, min=0.0, max=25.0, step=0.1, description='k21'),
    k23_val=FloatSlider(value=1.0, min=0.05, max=10.0, step=0.1, description='k23'),
    rangeX=FloatSlider(value=5.0, min=0.5, max=10.0, step=0.5),
    rangeY=FloatSlider(value=1.2, min=0.5, max=2.0, step=0.1)
);

<>:45: SyntaxWarning: invalid escape sequence '\c'
<>:45: SyntaxWarning: invalid escape sequence '\c'
/var/folders/zg/6xh1x_0x6cgc91tww6mq29200000gn/T/ipykernel_17583/3002412883.py:45: SyntaxWarning: invalid escape sequence '\c'
  f"initial slope ≈ {k12_val/(k21_val + k23_val):.2f} $M^{-1}\cdot s^{-1}$",


interactive(children=(IntSlider(value=3, description='linewidth', max=5, min=1), FloatSlider(value=5.0, descri…

<h3>Figure legend </h3><p>Concentration dependence (unit $M$) of the observed turnover rate (unit $s^{-1}$) according to the Briggs-Haldane model.   <span style="color:blue">Blue line</span>: Hyperbolic dependence of observed turnover rate vs concentration,  <span style="color:darkblue">dark blue line</span>: Slope of Briggs-Haldane hyperbola at concentration $c=0$, <span style="color:darkgray">dotted lines</span>: Crosshair at $c=K_{1/2}$ and $k_{obs} = V_{max}/2$. 
    <span style="color:blue;font-weight: bold">Note</span> that the triangle $k_{cat}/K_{1/2}$ reflects the inital slope, and may be interpreted (at first approximation only &#x26A0;) as substrate affinity.</p>
    <p><span style="color:blue">Observe</span> under which parameter combinations $K_{1/2}$ differs significantly from $K_S = \frac{k_{21}}{k_{12}}$!</p>
<img src="2-08_BriggsHaldane.png" width="466"> 